# Day 15: Testing and Data Engineering Intro
**PyTest and TestClient, tests drafted with Codex, JavaScript unit tests, ETL vs ELT, incremental loads and data quality**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q pytest fastapi httpx pandas openai tiktoken scikit-learn
!node --version

# ===== Option B: VS Code / local (run in your activated venv terminal) =====
# pip install pytest fastapi httpx pandas openai tiktoken scikit-learn
# node --version
import os
os.makedirs("day15", exist_ok=True)
os.chdir("day15")
print("Working in", os.getcwd())

## 2. PyTest basics

In [ ]:
%%writefile pricing.py
def with_gst(amount, rate=18):
    if amount < 0:
        raise ValueError("amount must not be negative")
    return round(amount * (1 + rate / 100), 2)

def grade(marks):
    if not 0 <= marks <= 100:
        raise ValueError("marks must be 0 to 100")
    if marks >= 85:
        return "A"
    if marks >= 70:
        return "B"
    if marks >= 50:
        return "C"
    return "Needs improvement"

In [ ]:
%%writefile test_pricing.py
import pytest
from pricing import with_gst, grade

def test_default_rate():
    assert with_gst(100) == 118.0

def test_negative_raises():
    with pytest.raises(ValueError):
        with_gst(-1)

@pytest.mark.parametrize("amount, rate, expected", [(100, 18, 118.0), (100, 5, 105.0), (0, 18, 0.0)])
def test_rates(amount, rate, expected):
    assert with_gst(amount, rate) == expected

@pytest.mark.parametrize("marks, expected", [(0, "Needs improvement"), (49, "Needs improvement"), (50, "C"),
                                             (69, "C"), (70, "B"), (84, "B"), (85, "A"), (100, "A")])
def test_grade_boundaries(marks, expected):
    assert grade(marks) == expected

def test_grade_out_of_range():
    with pytest.raises(ValueError):
        grade(101)

In [ ]:
# ===== Option A: Google Colab (active) =====
!python -m pytest -q test_pricing.py

# ===== Option B: VS Code =====
# Terminal: pytest -q    (or use the Testing panel, the flask icon in the sidebar)

## 3. Testing a FastAPI app with fixtures
A small SmartDesk-style tickets API.

In [ ]:
%%writefile tickets_api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI()
tickets: dict[int, dict] = {}

class TicketIn(BaseModel):
    title: str = Field(min_length=1)
    priority: str = Field(default="medium", pattern="^(low|medium|high)$")

@app.get("/tickets")
def list_tickets():
    return list(tickets.values())

@app.post("/tickets", status_code=201)
def create(t: TicketIn):
    tid = max(tickets, default=0) + 1
    tickets[tid] = {"id": tid, "title": t.title, "priority": t.priority, "status": "open"}
    return tickets[tid]

@app.get("/tickets/{tid}")
def get(tid: int):
    if tid not in tickets:
        raise HTTPException(404, "Ticket not found")
    return tickets[tid]

@app.delete("/tickets/{tid}", status_code=204)
def delete(tid: int):
    if tickets.pop(tid, None) is None:
        raise HTTPException(404, "Ticket not found")

In [ ]:
%%writefile test_tickets_api.py
import pytest
from fastapi.testclient import TestClient
import tickets_api

@pytest.fixture
def client():
    tickets_api.tickets.clear()          # every test starts with an empty store
    return TestClient(tickets_api.app)

def test_starts_empty(client):
    assert client.get("/tickets").json() == []

def test_create_returns_201(client):
    r = client.post("/tickets", json={"title": "No signal"})
    assert r.status_code == 201
    assert r.json() == {"id": 1, "title": "No signal", "priority": "medium", "status": "open"}

def test_empty_title_is_422(client):
    assert client.post("/tickets", json={"title": ""}).status_code == 422

def test_bad_priority_is_422(client):
    assert client.post("/tickets", json={"title": "x", "priority": "urgent!!"}).status_code == 422

def test_missing_is_404(client):
    assert client.get("/tickets/999").status_code == 404

def test_delete_twice(client):
    client.post("/tickets", json={"title": "x"})
    assert client.delete("/tickets/1").status_code == 204
    assert client.delete("/tickets/1").status_code == 404

In [ ]:
# ===== Option A: Google Colab (active) =====
!python -m pytest -q test_tickets_api.py

# ===== Option B: VS Code =====
# Terminal: pytest -q test_tickets_api.py

### TODO 1: Add a test that creating two tickets gives ids 1 and 2, and that GET /tickets returns both.

In [ ]:
# TODO: write your code here

## 4. NEW (v4): Generate tests with Codex, then review and run them
In VS Code you would ask OpenAI Codex for tests. Here, the cell below drafts tests with the LLM when a key is set, or loads a saved AI draft in offline mode. Either way, **you** review and run them.

The rules for `sla_minutes(priority)` are: high = 60, medium = 240, low = 1440 minutes. Any other priority raises `ValueError`. Priority is case-insensitive.

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

In [ ]:
%%writefile sla.py
def sla_minutes(priority):
    p = priority.strip().lower()
    table = {"high": 60, "medium": 240, "low": 1440}
    if p not in table:
        raise ValueError(f"unknown priority: {priority}")
    return table[p]

In [ ]:
RULES = """sla_minutes(priority): high = 60, medium = 240, low = 1440 minutes.
Any other priority raises ValueError. Priority is case-insensitive."""

SAVED_AI_DRAFT = """import pytest
from sla import sla_minutes

def test_high():
    assert sla_minutes("high") == 60

def test_medium():
    assert sla_minutes("medium") == 120

def test_low():
    assert sla_minutes("low") == 1440

def test_upper_case():
    assert sla_minutes("HIGH") == 60

def test_unknown():
    sla_minutes("urgent")

def test_returns_something():
    assert sla_minutes("low") is not None
"""

prompt = ("Write PyTest tests for this function. Rules:\n" + RULES +
          "\nUse plain asserts and pytest.raises. Return only Python code.\n\n" + open("sla.py").read())
draft = llm(prompt) if ONLINE else SAVED_AI_DRAFT
draft = draft.strip().removeprefix("```python").removesuffix("```").strip() + "\n"
open("test_sla_ai_draft.py", "w").write(draft)
print(f"Prompt tokens: {count_tokens(prompt)} | Draft saved ({'live model' if ONLINE else 'saved offline draft'})")
print(draft)

In [ ]:
# Run the AI draft as is. Failures are information, not a reason to edit the code.
!python -m pytest -q test_sla_ai_draft.py || true

### Review notes (offline draft)
| Test | Verdict | Reason |
|---|---|---|
| test_high, test_low, test_upper_case | keep | Match the rules. |
| test_medium | **fix** | The rule says 240, not 120. The test is wrong, the code is right. |
| test_unknown | **fix** | It calls the function without `pytest.raises`, so the expected ValueError makes the test fail. Wrap it in `pytest.raises`. |
| test_returns_something | **delete** | Checks nothing useful. |

Below is the reviewed version. Notice we did not change `sla.py`.

In [ ]:
%%writefile test_sla.py
import pytest
from sla import sla_minutes

@pytest.mark.parametrize("priority, minutes", [("high", 60), ("medium", 240), ("low", 1440),
                                               ("HIGH", 60), (" Low ", 1440)])
def test_rules(priority, minutes):
    assert sla_minutes(priority) == minutes

@pytest.mark.parametrize("bad", ["urgent", "", "p1"])
def test_unknown_priority_raises(bad):
    with pytest.raises(ValueError):
        sla_minutes(bad)

In [ ]:
!python -m pytest -q test_sla.py

## 5. JavaScript unit tests with Node's built-in runner
No installation needed. In a React project you would use Vitest and React Testing Library (see the deck); the ideas are the same.

In [ ]:
%%writefile tickets.mjs
export function formatMinutes(n) {
  return n >= 60 ? `${Math.floor(n / 60)} h ${n % 60} min` : `${n} min`;
}

export function openCount(tickets) {
  return tickets.filter((t) => t.status === "open").length;
}

In [ ]:
%%writefile tickets.test.mjs
import test from "node:test";
import assert from "node:assert/strict";
import { formatMinutes, openCount } from "./tickets.mjs";

test("formats minutes under an hour", () => {
  assert.equal(formatMinutes(45), "45 min");
});

test("formats hours and minutes", () => {
  assert.equal(formatMinutes(130), "2 h 10 min");
});

test("counts only open tickets", () => {
  assert.equal(openCount([{ status: "open" }, { status: "closed" }, { status: "open" }]), 2);
});

In [ ]:
# ===== Option A: Google Colab (active) =====
!node --test tickets.test.mjs

# ===== Option B: VS Code =====
# Terminal: node --test tickets.test.mjs

## 6. A tiny ETL pipeline: legacy ticket import
A deliberately messy export from the old helpdesk, the kind real systems produce.

In [ ]:
%%writefile legacy_tickets.csv
ticket_id,customer,minutes,opened,priority
1001, Asha ,25,2026-11-01,high
1002,Kenji,120,2026-11-01,medium
1002,Kenji,120,2026-11-01,medium
1003,Mei,,2026-11-02,low
1004,Ravi,abc,2026-11-02,medium
1005,Priya,56,02/11/2026,HIGH
1006, Hana,90,2026-11-03,low
1007,Arjun,310,2026-11-03,medium
1007,Arjun,310,2026-11-03,medium
1008,Yuki,45,2026-11-04,high
1009,Sara,78,2026-11-04,low
1010,Ken,15,2026-11-05,medium

In [ ]:
import pandas as pd, sqlite3

def extract(path):
    return pd.read_csv(path)

def transform(df):
    df = df.drop_duplicates("ticket_id").copy()
    df["customer"] = df["customer"].str.strip()
    df["priority"] = df["priority"].str.strip().str.lower()
    df["minutes"] = pd.to_numeric(df["minutes"], errors="coerce")
    df["opened"] = pd.to_datetime(df["opened"], format="mixed", dayfirst=True).dt.date.astype(str)
    return df.dropna(subset=["minutes"])

def load(df, conn, table="tickets"):
    df.to_sql(table, conn, if_exists="replace", index=False)
    return len(df)

raw = extract("legacy_tickets.csv")
clean = transform(raw)
conn = sqlite3.connect("smartdesk.db")
print("Raw rows:", len(raw), "-> Clean rows:", len(clean))
print("Loaded", load(clean, conn), "rows")
print(pd.read_sql("SELECT priority, COUNT(*) n, AVG(minutes) avg_min FROM tickets GROUP BY priority", conn))

## 7. The ELT version: load raw, transform with SQL

In [ ]:
raw.to_sql("raw_tickets", conn, if_exists="replace", index=False)
conn.executescript("""
DROP TABLE IF EXISTS tickets_clean;
CREATE TABLE tickets_clean AS
SELECT DISTINCT ticket_id, TRIM(customer) AS customer, CAST(minutes AS REAL) AS minutes, opened,
       LOWER(TRIM(priority)) AS priority
FROM raw_tickets
WHERE minutes IS NOT NULL AND CAST(minutes AS REAL) > 0;
""")
print("raw_tickets:", pd.read_sql("SELECT COUNT(*) n FROM raw_tickets", conn).n[0])
print("tickets_clean:", pd.read_sql("SELECT COUNT(*) n FROM tickets_clean", conn).n[0])

## 8. Idempotent incremental load

In [ ]:
def load_incremental(new_df, conn, table="tickets"):
    existing = set(pd.read_sql(f"SELECT ticket_id FROM {table}", conn)["ticket_id"])
    fresh = new_df[~new_df["ticket_id"].isin(existing)]
    fresh.to_sql(table, conn, if_exists="append", index=False)
    return len(fresh)

new_batch = pd.DataFrame([
    {"ticket_id": 1010, "customer": "Ken", "minutes": 15.0, "opened": "2026-11-05", "priority": "medium"},
    {"ticket_id": 1011, "customer": "Zara", "minutes": 99.0, "opened": "2026-11-06", "priority": "low"},
])
print("Run 1 inserted:", load_incremental(new_batch, conn))
print("Run 2 inserted:", load_incremental(new_batch, conn))
print("Total rows:", pd.read_sql("SELECT COUNT(*) n FROM tickets", conn).n[0])
conn.close()

## 9. Data quality tests on the loaded table

In [ ]:
%%writefile test_quality.py
import sqlite3
import pandas as pd
import pytest

@pytest.fixture
def tickets():
    with sqlite3.connect("smartdesk.db") as conn:
        return pd.read_sql("SELECT * FROM tickets", conn)

def test_ids_unique(tickets):
    assert tickets["ticket_id"].is_unique

def test_no_missing_minutes(tickets):
    assert tickets["minutes"].notna().all()

def test_minutes_not_negative(tickets):
    assert (tickets["minutes"] >= 0).all()

def test_priorities_valid(tickets):
    assert set(tickets["priority"]) <= {"low", "medium", "high"}

In [ ]:
!python -m pytest -q test_quality.py

### TODO 2 (SmartDesk step 13): add a freshness test (latest `opened` date is not older than 30 days from the newest ticket) and a test that the row count did not drop after the incremental load.

In [ ]:
# TODO: write your code here

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)